In [1]:
import os, csv, math, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import pyeeg as pe
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────────────────────
# 0. SETUP
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.getcwd())
BASE_DATA = os.path.join(PROJECT_ROOT, "data", "DEAP", "data_preprocessed_python")
OUT_FEAT  = os.path.join(PROJECT_ROOT, "data", "DEAP", "output", "features")
OUT_WORK  = os.path.join(PROJECT_ROOT, "data", "DEAP", "output", "work")
os.makedirs(OUT_FEAT, exist_ok=True)
os.makedirs(OUT_WORK, exist_ok=True)
print(f"BASE_DATA: {BASE_DATA}")
print(f"OUT_FEAT : {OUT_FEAT}")
print(f"OUT_WORK : {OUT_WORK}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────────────────
CHANNEL_IDX   = [0, 2, 3, 6, 7, 10, 11, 13, 16, 19, 20, 24, 25, 28, 29, 31]
CHANNEL_NAMES = [
    "Fp1","F3","F7","C3","T7","P3","P7","O1",
    "Fp2","F4","F8","C4","T8","P4","P8","O2",
]
BAND_EDGES  = [4, 8, 12, 16, 25, 45]
BAND_LABELS = ["Theta","Alpha","BetaL","BetaH","Gamma"]
WINDOW, STEP, FS = 256, 16, 128
SUBJECTS = [f"{i:02d}" for i in range(1, 21)]

N_CH    = len(CHANNEL_IDX)       # 16
N_BANDS = len(BAND_EDGES) - 1    # 5 
N_FEATS = N_BANDS * 2            # 10  (5 BP + 5 DE)s
FEAT_LABELS = (
    [f"BP_{b}" for b in BAND_LABELS] +
    [f"DE_{b}" for b in BAND_LABELS]
)

# ─────────────────────────────────────────────────────────────────────────────
# 2. FEATURE EXTRACTION  (skipped if already present from previous run)
# ─────────────────────────────────────────────────────────────────────────────
def differential_entropy(sig, band_edges, fs):
    de, fft_full = [], np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(len(sig), 1.0 / fs)
    for lo, hi in zip(band_edges[:-1], band_edges[1:]):
        mask = (freqs >= lo) & (freqs < hi)
        f = np.zeros_like(fft_full); f[mask] = fft_full[mask]
        var = np.var(np.fft.irfft(f, n=len(sig))) + 1e-10
        de.append(0.5 * np.log(2 * np.pi * np.e * var))
    return de

def extract_features(sub):
    src = os.path.join(BASE_DATA, f"s{sub}.dat")
    if not os.path.exists(src): return
    meta = []
    with open(src, "rb") as f:
        subject = pickle.load(f, encoding="latin1")
    for trial in range(40):
        data = subject["data"][trial]
        labels = np.asarray(subject["labels"][trial], dtype=np.float32)[:2]
        start = 0
        while start + WINDOW < data.shape[1]:
            feats = []
            for ch in CHANNEL_IDX:
                sig = data[ch][start: start + WINDOW]
                feats.extend(list(pe.bin_power(sig, BAND_EDGES, FS)[0]) +
                              differential_entropy(sig, BAND_EDGES, FS))
            meta.append([np.array(feats, dtype=np.float32), labels])
            start += STEP
    np.save(os.path.join(OUT_FEAT, f"s{sub}.npy"),
            np.array(meta, dtype=object), allow_pickle=True)

print("Checking / extracting features...")
for sub in tqdm(SUBJECTS, desc="Subjects"):
    if not os.path.exists(os.path.join(OUT_FEAT, f"s{sub}.npy")):
        extract_features(sub)

# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD + SPLIT + NORMALISE
# ─────────────────────────────────────────────────────────────────────────────
print("\nLoading data...")
all_X, all_L = [], []
found_subjects = 0
for sub in SUBJECTS:
    fp = os.path.join(OUT_FEAT, f"s{sub}.npy")
    if os.path.exists(fp):
        found_subjects += 1
        arr = np.load(fp, allow_pickle=True)
        for row in arr:
            label = np.asarray(row[1], dtype=np.float32).reshape(-1)
            if label.size >= 2:
                all_X.append(row[0])
                all_L.append(label[:2])

print(f"Feature files found for {found_subjects}/{len(SUBJECTS)} subjects")

all_X = np.array(all_X, dtype=np.float32)
all_L = np.array(all_L, dtype=np.float32)
if all_X.size == 0 or all_L.size == 0:
    raise RuntimeError(
        "No valid feature rows were loaded. Check BASE_DATA/OUT_FEAT paths and regenerate features if needed."
    )
if all_L.ndim != 2 or all_L.shape[1] < 2:
    raise RuntimeError(f"Unexpected label shape: {all_L.shape}. Expected (n_samples, >=2).")

y_arousal = (all_L[:, 0] > 6.5).astype(int)
y_valence = (all_L[:, 1] > 6.5).astype(int)
y_multi   = np.stack([y_arousal, y_valence], axis=1).astype(np.float32)
strat_key = [f"{a}{v}" for a, v in zip(y_arousal, y_valence)]

x_tr_raw, x_te_raw, y_tr, y_te = train_test_split(
    all_X, y_multi, test_size=0.2, random_state=SEED, stratify=strat_key)
print(f"Train {x_tr_raw.shape[0]:,}  |  Test {x_te_raw.shape[0]:,}")

scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr_raw).reshape(-1, N_CH, N_FEATS)
x_te = scaler.transform(x_te_raw).reshape(-1,  N_CH, N_FEATS)
with open(os.path.join(OUT_WORK, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

# ─────────────────────────────────────────────────────────────────────────────
# 4. CLASS-IMBALANCE WEIGHTS
# ─────────────────────────────────────────────────────────────────────────────
pw_aro = torch.tensor(
    [(y_tr[:, 0] == 0).sum() / max((y_tr[:, 0] == 1).sum(), 1)],
    dtype=torch.float32
).to(device)
pw_val = torch.tensor(
    [(y_tr[:, 1] == 0).sum() / max((y_tr[:, 1] == 1).sum(), 1)],
    dtype=torch.float32
).to(device)
print(f"pos_weight  Arousal={pw_aro.item():.3f}  Valence={pw_val.item():.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. DATASET
# ─────────────────────────────────────────────────────────────────────────────
class EEGDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):          return len(self.Y)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

# ─────────────────────────────────────────────────────────────────────────────
# 6. GAT LAYER
# ─────────────────────────────────────────────────────────────────────────────
class GATLayer(nn.Module):
    def __init__(self,
                 in_features:  int,
                 out_features: int,
                 num_heads:    int   = 4,
                 attn_dropout: float = 0.05,
                 residual:     bool  = True):
        super().__init__()
        self.H        = num_heads
        self.d        = out_features
        self.residual = residual

        self.W        = nn.Linear(in_features, num_heads * out_features, bias=False)
        self.a        = nn.Parameter(torch.empty(num_heads, 2 * out_features))
        nn.init.xavier_uniform_(self.a.unsqueeze(0))

        self.leaky     = nn.LeakyReLU(0.2)
        self.attn_drop = nn.Dropout(attn_dropout)   # FIX 3: light, attn only
        self.bn        = nn.BatchNorm1d(out_features)

        if residual:
            self.res_proj = (nn.Linear(in_features, out_features, bias=False)
                             if in_features != out_features else nn.Identity())

    def forward(self, x: torch.Tensor):
        B, N, _ = x.shape
        h    = self.W(x).view(B, N, self.H, self.d)
        hi   = h.unsqueeze(2)
        hj   = h.unsqueeze(1)
        pair = torch.cat([hi.expand(B, N, N, self.H, self.d),
                          hj.expand(B, N, N, self.H, self.d)], dim=-1)
        e     = self.leaky((pair * self.a.unsqueeze(0).unsqueeze(0).unsqueeze(0)).sum(-1))
        alpha = self.attn_drop(F.softmax(e, dim=2))            # (B, N, N, H)
        out   = torch.einsum("bqkh, bkhd -> bqhd", alpha, h).mean(dim=2)
        out   = self.bn(out.reshape(B * N, self.d)).reshape(B, N, self.d)
        out   = F.elu(out)
        if self.residual:
            out = out + self.res_proj(x)
        return out, alpha.permute(0, 3, 1, 2)                  # (B, H, N, N)

# ─────────────────────────────────────────────────────────────────────────────
# 7. TASK HEAD  (FIX 1: dense-only — no extra GAT layers)
# ─────────────────────────────────────────────────────────────────────────────
class TaskHead(nn.Module):
    def __init__(self, in_dim: int, dense: int, head_dropout: float = 0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, dense),
            nn.LayerNorm(dense),
            nn.GELU(),
            nn.Dropout(head_dropout),   # dropout only here (FIX 3)
            nn.Linear(dense, 1),
        )

    def forward(self, x):
        pooled = x.mean(dim=1)     # mean pool over channels → (B, in_dim)
        return self.head(pooled)   # (B, 1) raw logit

# ─────────────────────────────────────────────────────────────────────────────
# 8. FULL DEEP GAT MODEL
# ─────────────────────────────────────────────────────────────────────────────
class DeepGAT(nn.Module):
    def __init__(self,
                 n_channels:   int,
                 in_feats:     int,
                 backbone_dims: list,
                 dense_size:   int,
                 num_heads:    int   = 4,
                 attn_dropout: float = 0.05,
                 head_dropout: float = 0.3):
        super().__init__()

        # FIX 2: project raw features to backbone dim, then add channel embedding
        d0 = backbone_dims[0]
        self.input_proj = nn.Linear(in_feats, d0)

        # Learnable channel identity embedding — shape (1, N, d0)
        self.ch_embed = nn.Parameter(torch.randn(1, n_channels, d0) * 0.02)

        # Backbone GAT layers
        dims = [d0] + backbone_dims
        self.backbone = nn.ModuleList([
            GATLayer(dims[i], dims[i+1], num_heads, attn_dropout, residual=True)
            for i in range(len(backbone_dims))
        ])

        # Per-task dense heads
        self.head_aro = TaskHead(backbone_dims[-1], dense_size, head_dropout)
        self.head_val = TaskHead(backbone_dims[-1], dense_size, head_dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        # FIX 2: project + inject channel identity
        x = F.gelu(self.input_proj(x))   # (B, N, d0)
        x = x + self.ch_embed            # broadcast over batch

        backbone_attns = []
        for layer in self.backbone:
            x, aw = layer(x)
            backbone_attns.append(aw)    # (B, H, N, N)

        logit_aro = self.head_aro(x)     # (B, 1)
        logit_val = self.head_val(x)     # (B, 1)
        out = torch.cat([logit_aro, logit_val], dim=1)   # (B, 2) raw logits

        if return_attn:
            return out, backbone_attns
        return out

# ─────────────────────────────────────────────────────────────────────────────
# 9. LOSS
# ─────────────────────────────────────────────────────────────────────────────
class PerTaskWeightedBCE(nn.Module):
    def __init__(self, pw_aro: torch.Tensor, pw_val: torch.Tensor):
        super().__init__()
        self.loss_aro = nn.BCEWithLogitsLoss(pos_weight=pw_aro)
        self.loss_val = nn.BCEWithLogitsLoss(pos_weight=pw_val)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        return (self.loss_aro(logits[:, 0], targets[:, 0]) +
                self.loss_val(logits[:, 1], targets[:, 1])) / 2

# ─────────────────────────────────────────────────────────────────────────────
# 10. INSTANTIATE
# ─────────────────────────────────────────────────────────────────────────────
BACKBONE_DIMS = [64, 64, 64]   # FIX 1: 3 layers — full graph needs ≤3 hops
DENSE_SIZE    = 128
NUM_HEADS     = 4
ATTN_DROPOUT  = 0.05           # FIX 3: only on attention weights
HEAD_DROPOUT  = 0.3            # FIX 3: only in final MLP
BATCH_SIZE    = 256
EPOCHS        = 200
LR            = 5e-4

model = DeepGAT(
    n_channels    = N_CH,
    in_feats      = N_FEATS,
    backbone_dims = BACKBONE_DIMS,
    dense_size    = DENSE_SIZE,
    num_heads     = NUM_HEADS,
    attn_dropout  = ATTN_DROPOUT,
    head_dropout  = HEAD_DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters : {n_params:,}")
print(model)

criterion = PerTaskWeightedBCE(pw_aro, pw_val)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Linear warmup for 5 epochs, then cosine decay.
WARMUP_EPOCHS = 5
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS          # ramp 0 → 1
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))  # cosine decay 1 → 0

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

train_dl = DataLoader(EEGDataset(x_tr, y_tr), batch_size=BATCH_SIZE,
                      shuffle=True, drop_last=True)
test_dl  = DataLoader(EEGDataset(x_te, y_te), batch_size=BATCH_SIZE,
                      shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
# 11. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def accuracy(logits, targets):
    preds = (torch.sigmoid(logits) > 0.5).int()
    return (preds == targets.int()).float().mean().item()

csv_path = os.path.join(OUT_WORK, "training_log.csv")
with open(csv_path, "w", newline="") as f:
    csv.writer(f).writerow(
        ["epoch","train_loss","train_acc","val_loss","val_acc",
         "f1_arousal","f1_valence","f1_macro","lr"])

hist = {k: [] for k in
        ["train_loss","train_acc","val_loss","val_acc","f1_aro","f1_val"]}
best_f1 = 0.0

print("\nTraining Deep GAT (full graph, residual, per-task heads)...")
for epoch in range(1, EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    t_loss = t_acc = 0.0
    bar = tqdm(train_dl, desc=f"Ep {epoch:03d}/{EPOCHS}", leave=False)
    for Xb, Yb in bar:
        Xb, Yb = Xb.to(device), Yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss   = criterion(logits, Yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        t_acc  += accuracy(logits, Yb)
        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_t_loss = t_loss / len(train_dl)
    avg_t_acc  = t_acc  / len(train_dl)

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    v_loss = v_acc = 0.0
    all_p, all_t = [], []
    with torch.no_grad():
        for Xb, Yb in test_dl:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logits = model(Xb)
            v_loss += criterion(logits, Yb).item()
            v_acc  += accuracy(logits, Yb)
            preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()
            all_p.append(preds); all_t.append(Yb.cpu().numpy())

    avg_v_loss = v_loss / len(test_dl)
    avg_v_acc  = v_acc  / len(test_dl)
    P = np.vstack(all_p); T = np.vstack(all_t)
    f1_aro   = f1_score(T[:, 0], P[:, 0])
    f1_val   = f1_score(T[:, 1], P[:, 1])
    f1_macro = (f1_aro + f1_val) / 2

    if f1_macro > best_f1:
        best_f1 = f1_macro
        torch.save(model.state_dict(), os.path.join(OUT_WORK, "best_model.pth"))

    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    for k, v in zip(["train_loss","train_acc","val_loss","val_acc","f1_aro","f1_val"],
                    [avg_t_loss, avg_t_acc, avg_v_loss, avg_v_acc, f1_aro, f1_val]):
        hist[k].append(v)

    print(f"Ep {epoch:03d} | "
          f"T.Loss {avg_t_loss:.4f} | V.Loss {avg_v_loss:.4f} | "
          f"V.Acc {avg_v_acc:.4f} | F1-Aro {f1_aro:.4f} | "
          f"F1-Val {f1_val:.4f} | LR {lr_now:.2e}")

    with open(csv_path, "a", newline="") as f:
        csv.writer(f).writerow(
            [epoch, avg_t_loss, avg_t_acc, avg_v_loss, avg_v_acc,
             f1_aro, f1_val, f1_macro, lr_now])

print(f"\nBest macro F1 : {best_f1:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 12. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
model.load_state_dict(
    torch.load(os.path.join(OUT_WORK, "best_model.pth"), map_location=device))
model.eval()

all_p, all_t = [], []
with torch.no_grad():
    for Xb, Yb in test_dl:
        logits = model(Xb.to(device))
        preds  = (torch.sigmoid(logits) > 0.5).cpu().numpy()
        all_p.append(preds); all_t.append(Yb.numpy())
all_p = np.vstack(all_p); all_t = np.vstack(all_t)

for i, task in enumerate(["Arousal", "Valence"]):
    print(f"\n── {task} ──")
    print(classification_report(all_t[:, i], all_p[:, i],
                                 target_names=["Low","High"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, task in enumerate(["Arousal","Valence"]):
    cm = confusion_matrix(all_t[:, i], all_p[:, i])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Low","High"], yticklabels=["Low","High"], ax=axes[i])
    axes[i].set_title(f"Confusion Matrix — {task}")
    axes[i].set_ylabel("True"); axes[i].set_xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "confusion_matrices.png"), dpi=150)
plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 13. XAI — NATIVE GAT ATTENTION WEIGHTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[XAI] Extracting native GAT attention weights...")

x_xai = torch.tensor(x_te[:300]).float().to(device)
with torch.no_grad():
    _, bb_attns = model(x_xai, return_attn=True)

def mean_attn(aw_list):
    """Average over batch and heads → list of (N, N) arrays."""
    return [aw.mean(dim=(0, 1)).cpu().numpy() for aw in aw_list]

bb_maps = mean_attn(bb_attns)   # 3 × (16, 16)

# ── (a) Backbone: per-layer attention heatmaps ────────────────────────────
n_cols   = 3
n_rows   = math.ceil(len(bb_maps) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(7 * n_cols, 6 * n_rows))
if n_rows * n_cols == 1: axes = [axes]
else: axes = axes.flatten()

for li, mat in enumerate(bb_maps):
    sns.heatmap(mat, xticklabels=CHANNEL_NAMES, yticklabels=CHANNEL_NAMES,
                cmap="YlOrRd", vmin=0, annot=False, ax=axes[li],
                cbar=True, linewidths=0.2)
    axes[li].set_title(f"Backbone GAT Layer {li+1}\n(α_ij  averaged over heads + 300 samples)",
                        fontsize=11)
    axes[li].set_xlabel("Key channel (j)"); axes[li].set_ylabel("Query channel (i)")

for ax in axes[len(bb_maps):]:
    ax.axis("off")
plt.suptitle("Native GAT Attention — Backbone Layers", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "xai_backbone_layers.png"), dpi=150)
plt.close()
print("  Backbone layer plots saved.")

# ── (b) Channel importance evolution across depth ─────────────────────────
ch_imp_by_layer = np.array([mat.sum(axis=0) for mat in bb_maps])   # (3, 16)

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.plasma
for li in range(len(bb_maps)):
    color = cmap(li / max(len(bb_maps) - 1, 1))
    ax.plot(CHANNEL_NAMES, ch_imp_by_layer[li],
            marker="o", label=f"Layer {li+1}", color=color, linewidth=2)
ax.set_title(
    "Channel Attention Importance by Backbone Layer\n"
    "(Column-sum of α — how much each channel is 'attended to' per layer)",
    fontsize=13)
ax.set_xlabel("EEG Channel"); ax.set_ylabel("Cumulative Attention Received")
ax.legend(loc="upper right", fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "xai_channel_evolution.png"), dpi=150)
plt.close()

# ── (c) Branch comparison — Final Backbone Layer ──────────────────────────
final_mat = bb_maps[-1]
ch_imp    = final_mat.sum(axis=0)
order     = np.argsort(ch_imp)[::-1]
colors    = plt.cm.plasma(ch_imp[order] / ch_imp.max())

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar([CHANNEL_NAMES[i] for i in order], ch_imp[order],
        color=colors, edgecolor="black", lw=0.6)
ax.set_title(
    "Per-Channel Attention Importance (Final GAT Layer)\n"
    "Total attention received — directly from learned α weights",
    fontsize=13)
ax.set_xlabel("EEG Channel"); ax.set_ylabel("Cumulative Attention (α column-sum)")
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "xai_channel_importance_bar.png"), dpi=150)
plt.close()

print("  All attention XAI plots saved.")

# ─────────────────────────────────────────────────────────────────────────────
# 14. XAI — SHAP  (feature-level, per task)
# ─────────────────────────────────────────────────────────────────────────────
print("[XAI] Running SHAP (Arousal & Valence separately)...")

if torch.cuda.is_available():
    torch.backends.cudnn.enabled = False

class TaskWrapper(nn.Module):
    """Wraps model to output a single sigmoid probability for SHAP."""
    def __init__(self, base, task_idx):
        super().__init__()
        self.base = base; self.t = task_idx
    def forward(self, x):
        logits = self.base(x)
        return torch.sigmoid(logits[:, self.t]).unsqueeze(1)

bg   = torch.tensor(x_tr[:200]).float().to(device)
test = torch.tensor(x_te[:150]).float().to(device)

shap_data = {}
for t_idx, t_name in enumerate(["Arousal","Valence"]):
    wrapper   = TaskWrapper(model, t_idx).to(device)
    explainer = shap.GradientExplainer(wrapper, bg)
    sv        = explainer.shap_values(test)
    sv        = sv[0] if isinstance(sv, list) else sv
    while sv.ndim > 3:
        sv = sv.squeeze(-1)      # guarantee (150, N_CH, N_FEATS)

    mean_abs = np.abs(sv).mean(axis=0)
    shap_data[t_name] = {
        "heatmap":    mean_abs,
        "ch_total":   mean_abs.sum(axis=1),
        "feat_total": mean_abs.sum(axis=0),
    }

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(mean_abs,
                xticklabels=FEAT_LABELS, yticklabels=CHANNEL_NAMES,
                cmap="plasma", annot=True, fmt=".4f",
                linewidths=0.3, ax=ax)
    ax.set_title(
        f"SHAP Feature Importance — {t_name}\n"
        f"Left 5 = Band Power  |  Right 5 = Differential Entropy", fontsize=13)
    ax.set_xlabel("Feature (BP / DE per band)")
    ax.set_ylabel("EEG Channel")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_WORK, f"shap_heatmap_{t_name.lower()}.png"), dpi=150)
    plt.close()
    print(f"  SHAP heatmap saved: {t_name}")

# SHAP: Arousal vs Valence channel comparison
x_pos2 = np.arange(N_CH)
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_pos2 - 0.2, shap_data["Arousal"]["ch_total"], 0.38,
       label="Arousal", color="#C0392B", edgecolor="black")
ax.bar(x_pos2 + 0.2, shap_data["Valence"]["ch_total"], 0.38,
       label="Valence", color="#2980B9", edgecolor="black")
ax.set_xticks(x_pos2); ax.set_xticklabels(CHANNEL_NAMES, rotation=45)
ax.set_title("SHAP Channel Importance: Arousal vs Valence", fontsize=13)
ax.set_ylabel("Total |SHAP| per channel"); ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "shap_channel_aro_vs_val.png"), dpi=150)
plt.close()

# SHAP: Band Power vs Differential Entropy
bp_mean = np.mean([shap_data[t]["feat_total"][:N_BANDS]
                   for t in ["Arousal","Valence"]], axis=0)
de_mean = np.mean([shap_data[t]["feat_total"][N_BANDS:]
                   for t in ["Arousal","Valence"]], axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
xb = np.arange(N_BANDS)
ax.bar(xb - 0.2, bp_mean, 0.38, label="Band Power",
       color="#27AE60", edgecolor="black")
ax.bar(xb + 0.2, de_mean, 0.38, label="Differential Entropy",
       color="#8E44AD", edgecolor="black")
ax.set_xticks(xb); ax.set_xticklabels(BAND_LABELS)
ax.set_title("Feature Type SHAP: Band Power vs DE (averaged over tasks)", fontsize=13)
ax.set_ylabel("Mean |SHAP|"); ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_WORK, "shap_bp_vs_de.png"), dpi=150)
plt.close()

if torch.cuda.is_available():
    torch.backends.cudnn.enabled = True
print("  All SHAP plots saved.")

# ─────────────────────────────────────────────────────────────────────────────
# 15. TRAINING METRICS
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.3)

ax1 = fig.add_subplot(gs[0])
ax1.plot(hist["train_loss"], label="Train", color="#C0392B")
ax1.plot(hist["val_loss"],   label="Val",   color="#2980B9")
ax1.set_title("Loss  (Weighted BCE)")
ax1.legend()

ax2 = fig.add_subplot(gs[1])
ax2.plot(hist["train_acc"], label="Train", color="#C0392B")
ax2.plot(hist["val_acc"],   label="Val",   color="#2980B9")
ax2.set_title("Accuracy")
ax2.legend()

ax3 = fig.add_subplot(gs[2])
ax3.plot(hist["f1_aro"], label="F1 Arousal", color="#8E44AD")
ax3.plot(hist["f1_val"], label="F1 Valence", color="#E67E22")
ax3.set_title("F1 Score  (test set)")
ax3.legend()

plt.suptitle("Deep GAT v2 — Training Metrics", fontsize=15, y=1.02)
plt.savefig(os.path.join(OUT_WORK, "training_metrics.png"), dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"DONE — outputs in: {OUT_WORK}")
print("=" * 65)
outputs = [
    ("confusion_matrices.png",      "Arousal + Valence confusion matrices"),
    ("xai_backbone_layers.png",     "3 backbone GAT attention maps (α_ij)"),
    ("xai_channel_evolution.png",   "Channel importance across depth"),
    ("xai_channel_importance_bar.png","Bar chart of final backbone layer attention"),
    ("shap_heatmap_arousal.png",    "SHAP: channel × feature for Arousal"),
    ("shap_heatmap_valence.png",    "SHAP: channel × feature for Valence"),
    ("shap_channel_aro_vs_val.png", "SHAP channel comparison"),
    ("shap_bp_vs_de.png",           "Band Power vs DE importance"),
    ("training_metrics.png",        "Loss / Accuracy / F1 curves"),
    ("training_log.csv",            "Per-epoch log"),
    ("best_model.pth",              "Best checkpoint by macro F1"),
    ("scaler.pkl",                  "Fitted StandardScaler"),
]
for fname, desc in outputs:
    print(f"  {fname:<42} ← {desc}")

c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
BASE_DATA: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python
OUT_FEAT : c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\output\features
OUT_WORK : c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\output\work
Checking / extracting features...


Subjects: 100%|██████████| 20/20 [00:00<00:00, 20010.99it/s]


Loading data...
Feature files found for 0/20 subjects


RuntimeError: No valid feature rows were loaded. Check BASE_DATA/OUT_FEAT paths and regenerate features if needed.

In [2]:
# Helper cell: regenerate feature .npy files (safe, notebook-local)
import os, pickle
import numpy as np
from tqdm import tqdm
import pyeeg as pe

PROJECT_ROOT = os.path.abspath(os.getcwd())
BASE_DATA = os.path.join(PROJECT_ROOT, "data", "DEAP", "data_preprocessed_python")
OUT_FEAT  = os.path.join(PROJECT_ROOT, "data", "DEAP", "output", "features")
os.makedirs(OUT_FEAT, exist_ok=True)

CHANNEL_IDX   = [0, 2, 3, 6, 7, 10, 11, 13, 16, 19, 20, 24, 25, 28, 29, 31]
BAND_EDGES  = [4, 8, 12, 16, 25, 45]
WINDOW, STEP, FS = 256, 16, 128
SUBJECTS = [f"{i:02d}" for i in range(1, 21)]

def differential_entropy(sig, band_edges, fs):
    de, fft_full = [], np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(len(sig), 1.0 / fs)
    for lo, hi in zip(band_edges[:-1], band_edges[1:]):
        mask = (freqs >= lo) & (freqs < hi)
        f = np.zeros_like(fft_full); f[mask] = fft_full[mask]
        var = np.var(np.fft.irfft(f, n=len(sig))) + 1e-10
        de.append(0.5 * np.log(2 * np.pi * np.e * var))
    return de

def extract_features(sub):
    src = os.path.join(BASE_DATA, f"s{sub}.dat")
    if not os.path.exists(src):
        print(f"  missing source: {src}")
        return
    meta = []
    with open(src, "rb") as f:
        subject = pickle.load(f, encoding="latin1")
    for trial in range(40):
        data = subject["data"][trial]
        labels = np.asarray(subject["labels"][trial], dtype=np.float32)[:2]
        start = 0
        while start + WINDOW < data.shape[1]:
            feats = []
            for ch in CHANNEL_IDX:
                sig = data[ch][start: start + WINDOW]
                feats.extend(list(pe.bin_power(sig, BAND_EDGES, FS)[0]) +
                              differential_entropy(sig, BAND_EDGES, FS))
            meta.append([np.array(feats, dtype=np.float32), labels])
            start += STEP
    outp = os.path.join(OUT_FEAT, f"s{sub}.npy")
    np.save(outp, np.array(meta, dtype=object), allow_pickle=True)
    print(f"  wrote: {outp} ({len(meta)} rows)")

print('Regenerating features...')
for sub in tqdm(SUBJECTS, desc='Subjects'):
    extract_features(sub)
print('Regeneration complete. Files saved to:', OUT_FEAT)


Regenerating features...


Subjects: 100%|██████████| 20/20 [00:00<00:00, 19996.68it/s]

  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s01.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s02.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s03.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s04.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s05.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s06.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python\s07.dat
  missing source: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\dat

In [3]:
# Debug: show resolved paths used by kernel
print('PROJECT_ROOT ->', PROJECT_ROOT)
print('BASE_DATA   ->', BASE_DATA)
print('OUT_FEAT    ->', OUT_FEAT)
print('List OUT_FEAT exists?', os.path.exists(OUT_FEAT))
try:
    print('Listing OUT_FEAT:')
    print(os.listdir(OUT_FEAT))
except Exception as e:
    print('List error:', e)


PROJECT_ROOT -> c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap
BASE_DATA   -> c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\data_preprocessed_python
OUT_FEAT    -> c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\DeepGAT\notebooks_deap\data\DEAP\output\features
List OUT_FEAT exists? True
Listing OUT_FEAT:
[]


In [ ]:
# Rerun extraction using workspace-root DEAP data paths (correct location)
BASE_DATA = r"c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\data_preprocessed_python"
OUT_FEAT  = r"c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features"
os.makedirs(OUT_FEAT, exist_ok=True)
print('Using BASE_DATA ->', BASE_DATA)
print('Writing OUT_FEAT ->', OUT_FEAT)
for sub in tqdm(SUBJECTS, desc='Subjects (correct path)'):
    extract_features(sub)
print('Finished regenerating features at:', OUT_FEAT)


Using BASE_DATA -> c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\data_preprocessed_python
Writing OUT_FEAT -> c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features


Subjects (correct path):   5%|▌         | 1/20 [00:58<18:22, 58.03s/it]

  wrote: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features\s01.npy (19520 rows)


Subjects (correct path):  10%|█         | 2/20 [01:55<17:19, 57.74s/it]

  wrote: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features\s02.npy (19520 rows)


...

  wrote: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features\s10.npy (19520 rows)


Subjects (correct path):  55%|█████▌    | 11/20 [10:32<08:37, 57.48s/it]

  wrote: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features\s11.npy (19520 rows)


Subjects (correct path):  60%|██████    | 12/20 [11:29<07:39, 57.38s/it]

  wrote: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\features\s12.npy (19520 rows)
